In [1]:
import numpy as np 
import xarray as xr

In [2]:
import glob
filepath_daily_avg = '/lustre/storeB/project/fou/hi/foccus/datasets/norkystv3_averages/daily_avg'
ds_all_years = glob.glob(f'{filepath_daily_avg}/*/*.nc')

In [3]:
ds = xr.open_mfdataset(ds_all_years, engine='netcdf4')

In [5]:
ds

<xarray.Dataset> Size: 60GB
Dimensions:   (time: 4745, Y: 1148, X: 2747)
Coordinates:
    s_rho     float64 8B -0.004904
  * X         (X) float64 22kB 0.0 800.0 1.6e+03 ... 2.196e+06 2.197e+06
  * Y         (Y) float64 9kB 0.0 800.0 1.6e+03 ... 9.16e+05 9.168e+05 9.176e+05
    s_w       float64 8B 0.0
    lon       (Y, X) float64 25MB dask.array<chunksize=(1148, 2747), meta=np.ndarray>
    lat       (Y, X) float64 25MB dask.array<chunksize=(1148, 2747), meta=np.ndarray>
  * time      (time) datetime64[ns] 38kB 2012-01-05 2012-01-06 ... 2024-12-31
Data variables:
    salinity  (time, Y, X) float32 60GB dask.array<chunksize=(27, 1148, 2747), meta=np.ndarray>

In [6]:
#make sure its sorted 
ds = ds.sortby('time')

In [7]:
monthly_mean = ds.resample(time = 'M').mean('time')

/lustre/storeB/project/fou/hi/foccus/.venv/lib64/python3.11/site-packages/xarray/groupers.py:487: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


In [8]:
# burde nok heller ta det basert på måndetlige variasjoner? 
monthly_mean

<xarray.Dataset> Size: 2GB
Dimensions:   (time: 156, Y: 1148, X: 2747)
Coordinates:
    s_rho     float64 8B -0.004904
  * X         (X) float64 22kB 0.0 800.0 1.6e+03 ... 2.196e+06 2.197e+06
  * Y         (Y) float64 9kB 0.0 800.0 1.6e+03 ... 9.16e+05 9.168e+05 9.176e+05
    s_w       float64 8B 0.0
    lon       (Y, X) float64 25MB dask.array<chunksize=(1148, 2747), meta=np.ndarray>
    lat       (Y, X) float64 25MB dask.array<chunksize=(1148, 2747), meta=np.ndarray>
  * time      (time) datetime64[ns] 1kB 2012-01-31 2012-02-29 ... 2024-12-31
Data variables:
    salinity  (time, Y, X) float32 2GB dask.array<chunksize=(1, 1148, 2747), meta=np.ndarray>

In [9]:
ds_monthly = ds.groupby('time.month').mean('time')

In [10]:
ds_monthly

<xarray.Dataset> Size: 202MB
Dimensions:   (month: 12, Y: 1148, X: 2747)
Coordinates:
    s_rho     float64 8B -0.004904
  * X         (X) float64 22kB 0.0 800.0 1.6e+03 ... 2.196e+06 2.197e+06
  * Y         (Y) float64 9kB 0.0 800.0 1.6e+03 ... 9.16e+05 9.168e+05 9.176e+05
    s_w       float64 8B 0.0
    lon       (Y, X) float64 25MB dask.array<chunksize=(1148, 2747), meta=np.ndarray>
    lat       (Y, X) float64 25MB dask.array<chunksize=(1148, 2747), meta=np.ndarray>
  * month     (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
Data variables:
    salinity  (month, Y, X) float32 151MB dask.array<chunksize=(1, 1148, 2747), meta=np.ndarray>

In [54]:
symlink_path = f'/lustre/storeB/project/fou/hi/foccus/datasets/symlinks/norkystv3-hindcast/2024/' 
all_symlink = glob.glob(f'{symlink_path}/*')

In [ ]:
def calculate_anomalies(monthly_files, m_indx, output):
    ds = xr.open_mfdataset(monthly_files)
    monthly_baseline = ds_monthly.sel(month = m_indx)
    anomalies = ds - monthly_baseline
    anomalies.to_netcdf(output)  

In [18]:
dss = xr.open_mfdataset([f'/lustre/storeB/project/fou/hi/foccus/datasets/symlinks/norkystv3-hindcast/2024/norkyst800-20240101.nc', f'/lustre/storeB/project/fou/hi/foccus/datasets/symlinks/norkystv3-hindcast/2024/norkyst800-20240102.nc']).isel(s_w =-1, s_rho = -1)

In [19]:
salinity = dss.salinity

In [20]:
ano = salinity.values - ds_monthly.sel(month = 1).salinity.values

In [21]:
ano

array([[[        nan,         nan,         nan, ...,  0.01397705,
          0.00769043,  0.00904465],
        [        nan,         nan,         nan, ...,  0.03064728,
          0.02364349,  0.01139832],
        [        nan,         nan,         nan, ...,  0.03374481,
          0.02660751,  0.01397705],
        ...,
        [        nan,         nan,         nan, ..., -0.25105667,
         -0.25351715, -0.2541046 ],
        [        nan,         nan,         nan, ..., -0.2610817 ,
         -0.26238632, -0.26093292],
        [        nan,         nan,         nan, ..., -0.25093842,
         -0.2500496 , -0.25549316]],

       [[        nan,         nan,         nan, ...,  0.00697708,
          0.00469208,  0.00704575],
        [        nan,         nan,         nan, ...,  0.02964783,
          0.02264404,  0.0093956 ],
        [        nan,         nan,         nan, ...,  0.03574371,
          0.02660751,  0.0109787 ],
        ...,
        [        nan,         nan,         nan, ..., -